In [ ]:
# CELDA 001
'''
Relacionar google drive con Collab, para poder usar mi drive como almacenamiento de archivos,
y como lugar donde van a reposar los outputs que se generen en este script.
'''

from google.colab import drive
drive.mount('/content/t7drive')
# Ejecutar el código anterior y aceptar lo solicitado; permisos y accesos

In [ ]:
# CELDA 002
'''
Instalación de paquetes necesarios
'''
!pip install minisom # Paquete para desarrollar modelos SOM

In [3]:
# CELDA 003
'''
Importe de funciones, paquetes y librerías necesarias
'''
import os # Paquete para manejar carpetas direcciones y archivos
import pandas as pd # Paquete para manejar dataframes
import numpy as np # Paquete para manejar arreglos y matrices
import matplotlib.pyplot as plt # Paquete para manejar gráficos
import seaborn as sns # Paquete para mejorar la estetica de los graficos
from google.colab import files # Paquete para manejar archivos en google colab
from sklearn.decomposition import PCA # Paquete para manejar PCA
from sklearn.cluster import KMeans # Paquete para manejar K-means
from sklearn.metrics import silhouette_score # Paquete para manejar Silhouette
from sklearn.cluster import MiniBatchKMeans # Paquete para manejar Mini-Batch K-means
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster # Paquete para manejar dendrogramas
from sklearn.neighbors import NearestNeighbors # Paquete para manejar Nearest Neighbors
from sklearn.cluster import DBSCAN # Paquete para manejar DBSCAN
from sklearn.preprocessing import normalize # Paquete para manejar normalización de datos
from sklearn.metrics import silhouette_samples #

In [21]:
# CELDA 004
'''
Definir las rutas para guardar las figuras, tablas y notebooks.
'''

# Definir las rutas base
base = '/content/t7drive/MyDrive/taller7' # Debe crearse una carpeta directamente en Drive con el nombre de taller7
figs = os.path.join(base, 'figuras')
codes = os.path.join(base, 'codigos')
data = os.path.join(base, 'datos')

# Crear las rutas sino existen
os.makedirs(figs, exist_ok=True)
os.makedirs(codes, exist_ok=True)
os.makedirs(data, exist_ok=True)

In [ ]:
# CELDA 005
'''
Cargue de los datasets desde el dispositivo local:
- x_prueba
- x_entrena (se renombró porque originalmente se llamaba x_entena)
- y_entrena
'''

uploaded = files.upload()

for fn, content in uploaded.items():
  file_path = os.path.join(data, fn)
  # Check if file exists and remove it before saving
  if os.path.exists(file_path):
    os.remove(file_path)
    print(f'Removed existing file: "{fn}"')

  with open(file_path, 'wb') as f:
    f.write(content)
  print('User uploaded file "{name}" with length {length} bytes and saved to {path}'.format(
      name=fn, length=len(content), path=file_path))

  # Read the saved CSV file into a pandas DataFrame and assign to a variable
  try:
    if 'x_entrena' in fn:
      x_entrena_df = pd.read_csv(file_path)
      print(f'File "{fn}" loaded into DataFrame x_entrena_df.')
    elif 'x_prueba' in fn:
      x_prueba_df = pd.read_csv(file_path)
      print(f'File "{fn}" loaded into DataFrame x_prueba_df.')
    elif 'y_entrena' in fn:
      y_entrena_df = pd.read_csv(file_path)
      print(f'File "{fn}" loaded into DataFrame y_entrena_df.')
    else:
      print(f'File "{fn}" was uploaded but not assigned to a specific DataFrame variable.')

  except Exception as e:
    print(f'Error loading file "{fn}" into DataFrame: {e}')

# The dataframes are now available as x_entrena_df, x_prueba_df, and y_entrena_df

In [ ]:
# CELDA 006
# Visualizar la frecuencia de las clases del vector de etiquetas

class_counts = y_entrena_df.iloc[:, 0].value_counts()

plt.bar(class_counts.index, class_counts.values)
plt.xlabel('Clase')
plt.ylabel('Frecuencia')
plt.xticks(class_counts.index) # Ensure x-axis ticks are at the class values

# Add the count of records on top of each bar
for i, count in enumerate(class_counts.values):
    plt.text(class_counts.index[i], count + 5, str(count), ha='center') # Adjust the vertical position by adding 5

fig_path = os.path.join(figs, 'dist_clase.png')
plt.savefig(fig_path)
print(f"Class frequency bar plot saved to: {fig_path}")

plt.show()

In [ ]:
# CELDA 007
# Concatenar x_prueba_df y x_entrena_df

x_total = pd.concat([x_entrena_df, x_prueba_df]).reset_index(drop=True)
display(x_total.head())
print(f"Shape of combined DataFrame: {x_total.shape}")

In [ ]:
# CELDA 008
'''
Filtrado de atípicos utilizando el método IQR
'''

# Calculate Q1 and Q3 for each column
Q1 = x_total.quantile(0.25)
Q3 = x_total.quantile(0.75)
IQR = Q3 - Q1

# Define the bounds for outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter out the outliers
x_total_filtrado = x_total[~((x_total < lower_bound) | (x_total > upper_bound)).any(axis=1)]

print(f"Original shape of x_total: {x_total.shape}")
print(f"Shape of x_total after outlier filtering: {x_total_filtrado.shape}")


In [ ]:
# CELDA 009
# Generar matriz de correlación y guardar correlaciones
correlation_matrix = x_total_filtrado.corr(method='pearson')
correlation_df = pd.DataFrame(correlation_matrix)

# Visualizar la matriz de correlación
import seaborn as sns
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', fmt=".2f")
fig_path = os.path.join(figs, 'correlacion_heatmap.png')
plt.savefig(fig_path)
print(f"Correlation heatmap saved to: {fig_path}")
plt.show()

In [ ]:
# CELDA 010
'''
Análisis de componentes principales (PCA) y mapeo con y_entrena
'''

# Instantiate PCA and fit to the data
pca = PCA(n_components=None) # Retain 100% of variance by keeping all components
pca.fit(x_total_filtrado)

# Transform the data
combined_x_pca = pca.transform(x_total_filtrado)

print(f"Number of components selected: {pca.n_components_}")
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")

# Convert the PCA transformed data back to a DataFrame to easily merge with y_entrena_df
combined_x_pca_df = pd.DataFrame(combined_x_pca, index=x_total_filtrado.index)

In [ ]:
# CELDA 011
# Scree plot to visualize explained variance
plt.figure(figsize=(10, 6))
plt.plot(range(1, pca.n_components_ + 1), pca.explained_variance_ratio_, marker='o', linestyle='--')
plt.xlabel('Número de componentes')
plt.ylabel('Varianza explicada')
plt.grid(True)
# Save the scree plot
scree_plot_path = os.path.join(figs, 'pca_varianza.png')
plt.savefig(scree_plot_path)
print(f"Scree plot saved to: {scree_plot_path}")
plt.show()

# Scatter plot of the first two principal components
plt.figure(figsize=(10, 8))
plt.scatter(combined_x_pca[:, 0], combined_x_pca[:, 1])
plt.title('Scatter Plot of First Two Principal Components')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
# Save the scatter plot
scatter_plot_path = os.path.join(figs, 'scatter_plot_pc1pc2.png')
plt.savefig(scatter_plot_path)
print(f"Scatter plot saved to: {scatter_plot_path}")
plt.show()

In [ ]:
# CELDA 012
'''
Visualizar la distribución de cada variable en x_total_filtrado
'''

x_total_filtrado.hist(figsize=(15, 10))
plt.tight_layout()

# Save the figure to the 'figs' directory
fig_path = os.path.join(figs, 'variables_distribution.png')
plt.savefig(fig_path)
print(f"Histograms saved to: {fig_path}")

plt.show()

In [ ]:
# CELDA 013
'''
Generar un modelo de K-means con las siguientes características:

1. Probar en el rango de k(número de clusters) = 2 a 20.
2. Los centroides iniciales de los clusters se determinan con el
   algoritmo k-means++.
3. Los centroides se reiniciaran 10 veces, es decir, 10 corridas. n_init = 10
4. A partir de la semilla plantada que elige los centroides iniciales de cada
   iteración, ajecutar el algoritmo de LLoyd (asignar datos a un centroide y
   recalcular la media de las distancias en ese cluster) 100 veces. max_iter = 100.
   Poner adicionalmente un tope de iteración cuando ya la función objetivo (
   inercia) ya no se ajuste mas de 0.0001 con respecto a la iteración anterior.
   tol = 1e-4.
5. Capturar para el número de cluster que se está probando, la inercia al final
   de cada corrida (es decir, al final de un ciclo de iteracion), con la intención
   de tener la 'mejor' inercia para cada k probado. Y poder gráficar número de k
   vs inercia (Gráfico del codo).
6. Generar gráfico (o gráficos) de Silhoutte para comparar que k agrupa mejor los
   datos
'''

# Prepare data for K-Means. Using x_total_filtrado which is already processed.
# If you prefer to use the PCA transformed data, change x_total_filtrado to combined_x_pca
data_for_kmeans = x_total_filtrado

# Lists to store inertia and silhouette scores
inertia_scores = []
silhouette_scores = []
k_range = range(2, 21) # Range of k from 2 to 20

for k in k_range:
    # Initialize KMeans with specified parameters
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, max_iter=100, tol=1e-4, random_state=42)

    # Fit KMeans to the data
    kmeans.fit(data_for_kmeans)

    # Append the inertia to the list
    inertia_scores.append(kmeans.inertia_)

    # Calcular Silhouette
    silhouette_avg = silhouette_score(data_for_kmeans, kmeans.labels_)
    silhouette_scores.append(silhouette_avg)

# Plotting the Elbow Method (Inertia vs. k)
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia_scores, marker='o', linestyle='--')
plt.xlabel('Número de Clusters (k)')
plt.ylabel('Inercia')
plt.grid(True)
# Save the elbow plot
elbow_plot_path = os.path.join(figs, 'kmeans_elbow.png')
plt.savefig(elbow_plot_path)
print(f"Elbow method plot saved to: {elbow_plot_path}")
plt.show()

# Plotting the Silhouette Scores vs. k
if len(silhouette_scores) > 0:
    ks = list(k_range)  # [2, 3, ..., 20]  → mismo largo que silhouette_scores
    plt.figure(figsize=(10, 6))
    plt.plot(ks, silhouette_scores, marker='o', linestyle='--')
    plt.xlabel('Número de Clusters (k)')
    plt.ylabel('Coeficiente de Silhouette Promedio')
    plt.grid(True)
    silhouette_plot_path = os.path.join(figs, 'kmeans_silhouette.png')
    plt.savefig(silhouette_plot_path)
    print(f"Silhouette score plot saved to: {silhouette_plot_path}")
    plt.show()


In [ ]:
# CELDA 014
'''
Generar protoclústeres para reducir complejidad antes del agrupamiento jerárquico.
Se tienen las siguientes características

1. Método: Mini-Batch K-Means.
2. Semilla: random_state = 42.
3.Valores de m a probar: {400, 600, 800, 1000, 1200}.
4. Entrenamiento por m: n_init = 10, max_iter = 100, tol = 1e-4, batch_size ∈
{1024, 2048} (usar el mayor que soporte Colab).

5. Capturar por cada m:
	- proto_labels (longitud 9782): id de protoclúster por fila.
	- proto_sizes (longitud m): tamaño de cada protoclúster.
	- centroids (m×20): centroide de cada protoclúster.

6. Calidad: calcular coeficiente de Silhouette (distancia euclidiana) usando
todas las filas con sus proto_labels.
7. Gráfico: generar figura “m vs. Silhouette” para comparar los cinco valores de
m y seleccionar el m inicial para el jerárquico. Guardar la figura en la ruta
contenida en la variable figs.
'''

# Prepare data for Mini-Batch K-Means. Using x_total_filtrado.
data_for_protoclustering = x_total_filtrado

# Values of m (number of protoclusters) to test
m_values = [400, 600, 800, 1000, 1200]

# Lists to store silhouette scores for each m
silhouette_scores_protocluster = []

# Batch size to use (try the largest that Colab supports)
# Common large batch sizes: 1024, 2048. Let's start with 2048 and adjust if needed.
batch_size_to_use = 2048

# Dictionaries to store results for each m
proto_labels_dict = {}
proto_sizes_dict = {}
centroids_dict = {}


print(f"Using batch size: {batch_size_to_use}")

for m in m_values:
    print(f"\nRunning Mini-Batch K-Means for m = {m}...")
    # Initialize MiniBatchKMeans with specified parameters
    mbkmeans = MiniBatchKMeans(n_clusters=m,
                               init='k-means++',
                               n_init=10,
                               max_iter=100,
                               tol=1e-4,
                               batch_size=batch_size_to_use,
                               random_state=42,
                               verbose=0) # Set verbose to 0 to reduce output during fitting

    # Fit MiniBatchKMeans to the data
    mbkmeans.fit(data_for_protoclustering)

    # Capture results
    proto_labels = mbkmeans.labels_
    proto_sizes = np.bincount(proto_labels) # Count the number of data points in each protocluster
    centroids = mbkmeans.cluster_centers_

    # Store results in dictionaries
    proto_labels_dict[m] = proto_labels
    proto_sizes_dict[m] = proto_sizes
    centroids_dict[m] = centroids

    # Calculate silhouette score
    silhouette_avg = silhouette_score(data_for_protoclustering, proto_labels)
    silhouette_scores_protocluster.append(silhouette_avg)
    print(f"Silhouette Score for m = {m}: {silhouette_avg}")

# Plotting the Silhouette Scores vs. m
plt.figure(figsize=(10, 6))
plt.plot(m_values, silhouette_scores_protocluster, marker='o', linestyle='--')
plt.xlabel('Número de Protoclústeres (m)')
plt.ylabel('Coeficiente de Silhouette Promedio')
plt.grid(True)

# Save the silhouette plot for protoclustering
silhouette_protocluster_plot_path = os.path.join(figs, 'protocluster_silhouette.png')
plt.savefig(silhouette_protocluster_plot_path)
print(f"Protocluster Silhouette score plot saved to: {silhouette_protocluster_plot_path}")
plt.show()


In [ ]:
# CELDA 015
'''
Construir agrupamiento jerárquico sobre los centroides (m=400), evaluar k {2,…,12}
con Silhouette (euclidiana) calculado sobre todas las filas originales (vía propagación),
y generar un dendrograma truncado para la mejor configuración de cada método de
enlace. Construir con las siguientes condiciones:

1. m = 400.
2. Semilla: random_state = 42 (fijar también semillas globales para reproducibilidad).
3. Métodos de enlace a probar:
	- ward (euclidiana),
	- single (euclidiana),
	- average (UPGMA),
	- complete (euclidiana).

2. Rango de k: enteros de 2 a 12 (inclusive).
4. Bucle de evaluación para cada (método, k):
	a) construir el árbol jerárquico sobre centroids;
	b) cortar en ese k y obtener etiquetas para los 400 centroides;
	c) propagar esas etiquetas a las 9.782 filas usando proto_labels;
	d) calcular Silhouette (euclidiana) sobre todas las filas con esas 	etiquetas;
	e) guardar el valor de Silhouette y la tupla (método, k) en una estructura 	de resultados.

5. Al final, imprimir un resumen ordenado por Silhouette por método y global.
6. Generar dendrogramas (uno por método), para cada método, tomar la mejor (k, método)
según Silhouette y generar el dendrograma del árbol correspondiente sobre centroids,
truncado a los últimos 100 grupos (p.ej., lastp=100) sin labels en el eje X.
7. Guardar cada figura en la carpeta indicada por figs con nombres claros, p.ej.:
	-figs/dendro_ward_best.png
	-figs/dendro_single_best.png
	-figs/dendro_average_best.png
	-figs/dendro_complete_best.png.
'''

# Set global random state for reproducibility
np.random.seed(42)

# 1. m = 400. We will use the centroids generated for m=400 from the previous step.
m_to_use = 400
try:
    centroids_to_cluster = centroids_dict[m_to_use]
    proto_labels_for_propagation = proto_labels_dict[m_to_use]
    print(f"Using centroids for m = {m_to_use}")
except KeyError:
    print(f"Error: Centroids for m = {m_to_use} not found. Please run the previous cell (CELDA 015) first.")
    # Exit if centroids are not available
    raise

# 3. Linkage methods to test
linkage_methods = ['ward', 'single', 'average', 'complete']

# 4. Range of k
k_range_hierarchical = range(2, 13) # Inclusive of 12

# Structure to store results: list of tuples (method, k, silhouette_score)
hierarchical_results = []

# Data for Silhouette calculation (original data before protoclustering)
data_for_silhouette = x_total_filtrado # Use the filtered original data

print("\nEvaluating hierarchical clustering...")

for method in linkage_methods:
    print(f"\nProcessing linkage method: {method}")
    # a) Build the hierarchical tree on centroids
    Z = linkage(centroids_to_cluster, method=method, metric='euclidean')

    for k in k_range_hierarchical:
        print(f"  Evaluating k = {k}")
        # b) Cut the tree and get labels for the 400 centroids
        centroid_labels = fcluster(Z, k, criterion='maxclust')

        # c) Propagate these labels to the 9,782 original rows
        # Create a mapping from protocluster label to hierarchical cluster label
        proto_to_hierarchical_map = {i: centroid_labels[i] for i in range(m_to_use)}

        # Propagate labels: for each original data point, find its protocluster label,
        # then get the corresponding hierarchical cluster label.
        propagated_labels = np.array([proto_to_hierarchical_map[proto_label] for proto_label in proto_labels_for_propagation])

        # d) Calculate Silhouette score on the original data with propagated labels
        # Check if the number of unique labels is greater than 1 for silhouette score calculation
        if len(np.unique(propagated_labels)) > 1:
            silhouette_avg = silhouette_score(data_for_silhouette, propagated_labels, metric='euclidean')
            print(f"    Silhouette Score for k = {k}: {silhouette_avg:.4f}")
            # e) Store the result
            hierarchical_results.append((method, k, silhouette_avg))
        else:
            print(f"    Skipping Silhouette calculation for k = {k} as only one cluster was formed.")
            hierarchical_results.append((method, k, np.nan)) # Store NaN if silhouette cannot be calculated


# 5. Print summary ordered by Silhouette
print("\nSummary of Hierarchical Clustering Results (ordered by Silhouette Score):")

# Global summary
global_summary = sorted(hierarchical_results, key=lambda item: item[2] if not np.isnan(item[2]) else -np.inf, reverse=True)
print("\nGlobal Ranking:")
for method, k, silhouette in global_summary:
     print(f"  Method: {method}, k: {k}, Silhouette: {silhouette:.4f}" if not np.isnan(silhouette) else f"  Method: {method}, k: {k}, Silhouette: N/A")

# Summary by method
print("\nRanking by Method:")
for method in linkage_methods:
    method_results = sorted([res for res in hierarchical_results if res[0] == method],
                            key=lambda item: item[2] if not np.isnan(item[2]) else -np.inf,
                            reverse=True)
    print(f"\n  Method: {method}")
    for m, k, silhouette in method_results:
         print(f"    k: {k}, Silhouette: {silhouette:.4f}" if not np.isnan(silhouette) else f"    k: {k}, Silhouette: N/A")


# 6. Generate truncated dendrograms for the best configuration of each method
print("\nGenerating dendrograms for the best configuration of each method...")

for method in linkage_methods:
    # Find the best k for this method based on Silhouette score
    best_result_for_method = max([res for res in hierarchical_results if res[0] == method and not np.isnan(res[2])],
                                 key=lambda item: item[2], default=None)

    if best_result_for_method:
        best_method, best_k, best_silhouette = best_result_for_method
        print(f"  Best configuration for method '{best_method}': k = {best_k}, Silhouette = {best_silhouette:.4f}")

        # Re-calculate linkage for the best configuration
        Z_best = linkage(centroids_to_cluster, method=best_method, metric='euclidean')

        # Generate truncated dendrogram
        plt.figure(figsize=(12, 8))
        dendrogram(Z_best,
                   truncate_mode='lastp', # Show only the last p merged clusters
                   p=100,                 # Show the last 100 merged clusters
                   show_leaf_counts=False,
                   show_contracted=True,  # To show the original number of points in the contracted branches
                   no_labels=None            # Do not show labels on the x-axis
                  )
        #plt.title(f'Dendrograma enlace: ({best_method} linkage, mejor k = {best_k})')
        plt.xlabel('Tamaño del grupo') # Set the desired x-axis label
        plt.ylabel('Distancia')
        plt.grid(False)
        ax = plt.gca()
        ax.set_xticks([])
        ax.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)

        # Save the figure
        dendrogram_path = os.path.join(figs, f'dendro_{best_method}_best.png')
        plt.savefig(dendrogram_path)
        print(f"  Dendrogram saved to: {dendrogram_path}")
        plt.show()
    else:
        print(f"  No valid results found for method '{method}' to generate a dendrogram.")

In [ ]:
# CELDA 016
'''
Elección de epsilon (eps) para posterior agrupamiento con método DBSCAN.
Calcular la curva k-dist y generar k-plot (grafico del codo) para diferentes
valores de min_samples.
'''

# Data to use for k-distance analysis
data_for_kdist = x_total_filtrado
'''
X_cos = normalize(x_total_filtrado, norm='l2', axis=1)
data_for_kdist = X_cos
'''
# Values of min_samples to test
min_samples_values = [8, 10, 12, 16, 20]

print("Calculating k-distance curves for different min_samples values...")

for min_samples in min_samples_values:
    print(f"\nProcessing min_samples = {min_samples}")

    # Find the k-nearest neighbors (k = min_samples)
    # The number of neighbors to consider is min_samples + 1 because the point itself is included in the distances.
    neigh = NearestNeighbors(n_neighbors=min_samples, algorithm='auto', metric='euclidean')
    #neigh = NearestNeighbors(n_neighbors=min_samples, algorithm='auto', metric='cosine')
    neigh.fit(data_for_kdist)

    # Find the distances to the k-nearest neighbors
    distances, indices = neigh.kneighbors(data_for_kdist)

    # Get the distance to the k-th nearest neighbor (the last column)
    k_distance = distances[:, min_samples - 1] # Index is min_samples - 1 because distances include the point itself

    # Sort the distances in ascending order
    k_distance_sorted = np.sort(k_distance)

    # Plot the k-distance curve
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(k_distance_sorted) + 1), k_distance_sorted)
    plt.xlabel('Puntos de datos (ordenados por distancia)')
    plt.ylabel(f'Distancia')
    #plt.title(f'Curva de k-distancia para min_samples = {min_samples}')
    plt.grid(True)

    # Save the k-distance plot
    kdist_plot_path = os.path.join(figs, f'kdist_plot_min_samples_{min_samples}.png')
    plt.savefig(kdist_plot_path)
    print(f"k-distance plot saved to: {kdist_plot_path}")

    plt.show()

print("\nK-distance curve calculation and plotting complete.")

In [ ]:
# CELDA 017
'''
Ejecutar DBSCAN con las siguientes condiciones

0. Usar métrica euclidiana
1. Para cada ejecución registrar:
	- eps
	- min_samples
	- n_clusters (excluyendo -1)
	- noise_Ratio #(-1)/N
	- silhouette calculada solo con puntos ≠ -1(Si hay <2 clústeres válidos, guardar silhouette = NaN)

2. Probar min_samples entre 5, 8, 10, 12, 16
3. Para cada valor de min_samples probar el rango 4.5-5.0 de distancia (eps) variando 0.1 en cada iteración.
4. Imprimir una tabla ordenada con la siguiente información:
	- silhouette (desc)
	- noise_ratio (asc)
	- n_clusters (desc)
	- Imprimir Top-4 configuraciones (pero guardar todas).
	- Si hay empates, desempatar por Silhouette, eps o mayor min_samples, en ese orden.
5. Graficar silhouette para las mejores 4 configuraciones. Guardar gráficos en la ruta de la variable figs.
'''

# Data to use for DBSCAN
data_for_dbscan = x_total_filtrado
#data_for_dbscan = X_cos

# 2. Probar min_samples
min_samples_values_dbscan = [5, 8, 10, 12, 16]

# 3. Para cada valor de min_samples probar variando 0.1 en cada iteración.
eps_values = np.arange(4.5, 5.1, 0.1) # Use numpy.arange for floating-point steps

# Structure to store results: list of dictionaries
dbscan_results = []

print("Evaluating DBSCAN clustering with different eps and min_samples values...")

for min_samples in min_samples_values_dbscan:
    print(f"\nProcessing min_samples = {min_samples}")
    for eps in eps_values:
        print(f"  Evaluating eps = {eps:.2f}")

        # Ejecutar DBSCAN
        dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric='euclidean')
        #dbscan = DBSCAN(eps=eps, min_samples=int(min_samples), metric='cosine')
        dbscan.fit(data_for_dbscan)

        # Obtener las etiquetas de cluster
        labels = dbscan.labels_

        # Calcular n_clusters (excluyendo ruido)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

        # Calcular noise_Ratio #(-1)/N
        n_noise = list(labels).count(-1)
        noise_ratio = n_noise / len(data_for_dbscan)

        # Calcular silhouette calculada solo con puntos ≠ -1
        silhouette_avg = np.nan # Initialize with NaN

        # Check if there are at least 2 clusters (excluding noise) and more than 1 data point
        # to calculate silhouette score
        if n_clusters >= 2 and len(data_for_dbscan) > 1:
             # Filter out noise points (-1) for silhouette calculation
            sample_indices = np.where(labels != -1)[0]
            if len(sample_indices) > 1 and len(np.unique(labels[sample_indices])) > 1:
                silhouette_avg = silhouette_score(data_for_dbscan.iloc[sample_indices], labels[sample_indices])
                '''
                silhouette_avg = silhouette_score(
                    data_for_dbscan[sample_indices] if isinstance(data_for_dbscan, np.ndarray) else data_for_dbscan.iloc[sample_indices],
                    labels[sample_indices],
                    metric='cosine'
                    )

                Xsub = data_for_dbscan[sample_indices] if isinstance(data_for_dbscan, np.ndarray) else data_for_dbscan.iloc[sample_indices]
                silhouette_avg = silhouette_score(Xsub, labels[sample_indices], metric='cosine')
                '''
            else:
                 print(f"    Skipping Silhouette for eps = {eps:.2f}, min_samples = {min_samples}: Not enough valid clusters or data points.")


        print(f"    n_clusters: {n_clusters}, noise_ratio: {noise_ratio:.4f}, silhouette: {silhouette_avg:.4f}" if not np.isnan(silhouette_avg) else f"    n_clusters: {n_clusters}, noise_ratio: {noise_ratio:.4f}, silhouette: N/A")

        # Store the results
        dbscan_results.append({
            'eps': eps,
            'min_samples': min_samples,
            'n_clusters': n_clusters,
            'noise_ratio': noise_ratio,
            'silhouette': silhouette_avg
        })

# 4. Imprimir una tabla ordenada con la siguiente información:
results_df = pd.DataFrame(dbscan_results)

# Sort the results based on specified criteria
# Sort by silhouette (desc), then noise_ratio (asc), then n_clusters (desc), then eps (desc), then min_samples (desc)
results_df_sorted = results_df.sort_values(by=['silhouette', 'noise_ratio', 'n_clusters', 'eps', 'min_samples'],
                                            ascending=[False, True, False, False, False])


print("\nSummary of DBSCAN Clustering Results (ordered by Silhouette):")
# Print Top-4 configurations
print("\nTop 4 Configurations:")
display(results_df_sorted.head(4))

print("\nAll Results:")
display(results_df_sorted)

# 5. Graficar silhouette para las mejores 4 configuraciones.
print("\nGenerating silhouette plots for the top 4 configurations...")

# Select top 4 configurations
top_4_configs = results_df_sorted.head(4)

for index, row in top_4_configs.iterrows():
    eps = row['eps']
    min_samples = row['min_samples']
    silhouette_score_val = row['silhouette']

    if not np.isnan(silhouette_score_val):
        print(f"  Generating Silhouette plot for eps={eps:.4f}, min_samples={min_samples}")

        # Re-run DBSCAN for the selected configuration to get labels
        #dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric='euclidean')
        #dbscan = DBSCAN(eps=eps, min_samples=int(min_samples), metric='cosine')
        dbscan.fit(data_for_dbscan)
        labels = dbscan.labels_

        # Filter out noise points for silhouette analysis and plotting
        sample_indices = np.where(labels != -1)[0]
        if len(sample_indices) > 1 and len(np.unique(labels[sample_indices])) > 1:
            from sklearn.metrics import silhouette_samples
            silhouette_vals = silhouette_samples(data_for_dbscan.iloc[sample_indices], labels[sample_indices])
            '''
            Xsub = data_for_dbscan[sample_indices] if isinstance(data_for_dbscan, np.ndarray) else data_for_dbscan.iloc[sample_indices]
            silhouette_vals = silhouette_samples(Xsub, labels[sample_indices], metric='cosine')
            '''
            # Create silhouette plot
            plt.figure(figsize=(10, 6))
            y_lower = 10
            unique_labels = np.unique(labels[sample_indices])
            unique_labels.sort() # Sort labels for consistent plotting order

            for i in unique_labels:
                # Aggregate the silhouette scores for samples belonging to cluster i, and sort them
                ith_cluster_silhouette_values = silhouette_vals[labels[sample_indices] == i]
                ith_cluster_silhouette_values.sort()

                size_cluster_i = ith_cluster_silhouette_values.shape[0]
                y_upper = y_lower + size_cluster_i

                color = plt.cm.nipy_spectral(float(i) / len(unique_labels))
                plt.fill_betweenx(np.arange(y_lower, y_upper),
                                  0, ith_cluster_silhouette_values,
                                  facecolor=color, edgecolor=color, alpha=0.7)

                # Label the silhouette plots with their cluster numbers at the middle
                plt.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))

                # Compute the new y_lower for next plot
                y_lower = y_upper + 10  # 10 for the gap between clusters

            plt.xlabel("The silhouette coefficient values")
            plt.ylabel("Cluster label")
            plt.title(f"Silhouette Plot for DBSCAN (eps={eps:.2f}, min_samples={min_samples})")
            plt.axvline(x=silhouette_score_val, color="red", linestyle="--", label=f'Average Silhouette ({silhouette_score_val:.2f})')
            plt.yticks([])  # Clear the y-axis labels / ticks
            plt.xticks(np.arange(-1, 1.1, 0.1))
            plt.legend()
            plt.grid(True)

            # Save the silhouette plot
            silhouette_dbscan_plot_path = os.path.join(figs, f'dbscan_silhouette_eps_{eps:.2f}_min_samples_{min_samples}.png')
            plt.savefig(silhouette_dbscan_plot_path)
            print(f"  Silhouette plot saved to: {silhouette_dbscan_plot_path}")
            plt.show()
        else:
             print(f"  Skipping Silhouette plot for eps={eps:.2f}, min_samples={min_samples}: Not enough valid clusters or data points after filtering noise.")

    else:
        print(f"  Skipping Silhouette plot for eps={eps:.2f}, min_samples={min_samples}: Silhouette score is NaN.")

print("\nDBSCAN evaluation complete.")

In [ ]:
# CELDA 018
'''
Implementar un experimento de SOM (Self-Organizing Map) sobre el dataset
x_total_filtrado. Usar MiniSom o librería equivalente que soporte Batch SOM.
Teniendo en cuenta las siguientes especificaciones:

1. Fijar random_state = 42 y cualquier otra semilla relevante.
2. Tamaño de retícula, probar con tamaños nxn con n (16-36) variando de 1. Para
   cada tamaño entrenar un SOM independiente.
3. Función de vecindad: Gaussiana.
4. Plan de entrenamiento:
	- Fase Rough / ordenamiento global: 20 épocas con vecindad amplia.
	- Fase Fine-tuning / local: 80 épocas con vecindad corta.
	- Tasa de aprendizaje de 0.4 a 0.2
	- Radio de vecindad 0.5 x tamaño de la retícula. Decaimiento líneal por fase
	- Calcula el Quantization Error (QE) de cada modelo y guárdalo en una variable

5. Generar gráfico del QE vs Tamaño de retícula
6. Generar gráfica con el nombre umatrix_som_{m}x{n}.png para la configuración
   SOM con mejor QE.
7. Guardar en una variable y mostrarla en consola, una tabla que muestre el tamaño
   de la réticula, el número de neuronas y el QE, ordenar de forma ascendente según QE.
'''

# Import MiniSom
from minisom import MiniSom

# 1. Fijar random_state
random_state = 42
np.random.seed(random_state)

# Data to use for SOM (using the filtered data)
data_for_som = x_total_filtrado.values # MiniSom works with numpy arrays

# Scale the data to be between 0 and 1 for better SOM performance
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
# data_for_som_scaled = scaler.fit_transform(data_for_som)
data_for_som_scaled = x_total_filtrado.values

# 2. Tamaño de retícula, probar con tamaños nxn con n (16-36) variando de 1.
#n_values = range(16, 37) # n from 16 to 36 inclusive
n_values = [18, 22, 26, 30, 34]

# List to store SOM results
som_results = []

# Store QE for each model
quantization_errors = {}

print("Running SOM experiments with different grid sizes...")

for n in n_values:
    m = n # Square grid (nxn)
    grid_size = (n, m)
    num_neurons = n * m

    print(f"\nTraining SOM with grid size: {n}x{m} ({num_neurons} neurons)")

    # 4. Plan de entrenamiento:
    # Initialize MiniSom
    som = MiniSom(n, m, data_for_som_scaled.shape[1], sigma=0.5 * max(n, m), learning_rate=0.4,
                  neighborhood_function='gaussian', random_seed=random_state)

    # Initialize weights randomly
    som.random_weights_init(data_for_som_scaled)

    # --- Rough / ordenamiento global: 20 "épocas" reales ---
    L = max(n, m)
    num_epochs_rough = 20
    som.train_random(data_for_som_scaled, num_epochs_rough * len(data_for_som_scaled), verbose=False)

    # Ajustar hiperparámetros para Fine: radio corto y alpha menor
    som.sigma = 0.25 * L
    som.learning_rate = 0.2

    # --- Fine-tuning / local: 80 "épocas" reales ---
    num_epochs_fine = 80
    som.train_random(data_for_som_scaled, num_epochs_fine * len(data_for_som_scaled), verbose=False)

    # Calculate the Quantization Error (QE)
    qe = som.quantization_error(data_for_som_scaled)
    quantization_errors[grid_size] = qe
    print(f"  Quantization Error: {qe:.4f}")

    # Store results
    som_results.append({
        'grid_size': f'{n}x{m}',
        'num_neurons': num_neurons,
        'qe': qe,
        'som_model': som # Store the trained SOM model for later U-matrix plotting
    })

print("\nSOM experiments complete.")

# 5. Generar gráfico del QE vs tamaño de la retícula (n)
n_values_list = [res['grid_size'] for res in som_results]
qe_values = [res['qe'] for res in som_results]

plt.figure(figsize=(12, 6))
plt.plot(n_values, qe_values, marker='o', linestyle='-')
plt.xlabel('Tamaño de la retícula (n)')
plt.ylabel('Quantization Error (QE)')
# plt.title('Quantization Error vs. Grid Size')
plt.grid(True)

# Save the QE plot
qe_plot_path = os.path.join(figs, 'som_qe_vs_grid_size.png')
plt.savefig(qe_plot_path)
print(f"QE vs Grid Size plot saved to: {qe_plot_path}")
plt.show()


# 7. Guardar en una variable y mostrarla en consola, una tabla que muestre el tamaño de la réticula,
# el número de neuronas y el QE, ordenar de forma ascendente según QE.
results_df = pd.DataFrame(som_results)

# Sort by QE in ascending order
results_df_sorted = results_df.sort_values(by='qe', ascending=True)

print("\nSOM Results Summary (ordered by Quantization Error):")
display(results_df_sorted[['grid_size', 'num_neurons', 'qe']])

# 6. Generar gráfica con el nombre umatrix_som_{m}x{n}.png para la configuración SOM con mejor QE.
best_som_config = results_df_sorted.iloc[0]
best_grid_size_str = best_som_config['grid_size']
best_som_model = best_som_config['som_model']

print(f"\nGenerating U-matrix for the best SOM configuration ({best_grid_size_str})...")

# Get the U-matrix
umatrix = best_som_model.distance_map().T # Transpose for correct orientation

# Plot the U-matrix
plt.figure(figsize=(10, 10))
plt.pcolor(umatrix, cmap='bone_r')  # Use bone_r colormap for typical U-matrix visualization
plt.colorbar(label='Distancia de la neurona')
#plt.title(f'U-matrix for SOM ({best_grid_size_str})')
plt.axis('off') # Hide axes

# Save the U-matrix plot
umatrix_plot_path = os.path.join(figs, f'umatrix_som_{best_grid_size_str.replace("x", "x")}.png') # Replace 'x' to match the requested filename format
plt.savefig(umatrix_plot_path)
print(f"U-matrix plot saved to: {umatrix_plot_path}")
plt.show()

In [ ]:
# CELDA 019
'''
Para los dataframes x_total_filtrado y combined_x_pca_df (solo con las primeras
15 variables)  probar el mejor modelo k-means encontrado con las siguientes
características.

1. Para k in {2,3,4,5}, entrenar KMeans con:
	- init='k-means++'
	- n_init=10
	- max_iter=100
	- tol=1e-4
	- random_state=42.

2. Guardar para cada k:
	- inercia (SSE)
	- silhouette (si hay ≥2 clusters; usa todas las filas)

3. Genera dos gráficos (uno por dataset):
	- Codo: k vs inercia
	- Silhouette promedio: k vs silhouette

4. Elige el mejor k por silhouette (máximo). Con ese modelo:
	- Obtén labels de K-Means.
	- En las filas labeled (mask_labeled), asignar a cada clúster la etiqueta
    mayoritaria (voto mayoritario) de acuerdo al cruce de los dataframes
    implementados con y_entrena.

5. Evalúa sobre las filas etiquetadas:
  - Exactitud (accuracy) = proporción de aciertos
  - Precisión (precision macro) = promedio no ponderado de precisión por clase (usa average='macro').
  - Reporta también la matriz de confusión

6. Entregar un resumen (imprime pero guarda en una variable) final por dataset:
   mejor k, silhouette, inercia, accuracy, precision macro y el diccionario
   cluster -> etiqueta.
'''

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, accuracy_score, precision_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import os

# ============== Máscara de etiquetadas y etiquetas alineadas ==============
def build_mask_and_labels(x_filtrado: pd.DataFrame, y_entrena_df: pd.DataFrame | pd.Series):
    """
    Crea:
      - mask_labeled (n,): True si la fila proviene de x_entrena_df y sobrevivió al filtrado IQR.
      - y_train_full: Series (n,) con etiquetas en filas 'train' y NaN en 'test'.
    Supone:
      - x_total = concat([x_entrena_df, x_prueba_df]).reset_index(drop=True)
      - x_total_filtrado = x_total[mask_outliers]  (SIN reset_index).
    """
    n_train = len(y_entrena_df)

    # <-- FIX: esto ya es ndarray booleano; no uses .to_numpy()
    mask_labeled = (pd.Index(x_filtrado.index) < n_train)
    # (equivalentes: x_filtrado.index.values < n_train  o  np.asarray(x_filtrado.index) < n_train)

    y_vals = pd.Series(y_entrena_df.squeeze()).to_numpy()
    y_train_full = pd.Series(np.nan, index=x_filtrado.index, name="clase")

    # índices originales (< n_train) que sobrevivieron al filtrado
    survived_train_idx = x_filtrado.index[mask_labeled]

    # asigna etiquetas verdaderas en las filas de train que sobrevivieron
    y_train_full.loc[survived_train_idx] = y_vals[survived_train_idx]
    return mask_labeled, y_train_full

def majority_vote_mapping(y_full: pd.Series, cluster_labels: np.ndarray, mask_labeled: np.ndarray) -> dict:
    idx = np.where(mask_labeled)[0]
    dfm = pd.DataFrame({"c": cluster_labels[idx], "y": y_full.iloc[idx].values}).dropna(subset=["y"])
    if dfm.empty:
        return {}
    return dfm.groupby("c")["y"].apply(lambda s: s.value_counts().idxmax()).to_dict()

def eval_on_mask(y_full: pd.Series, cluster_labels: np.ndarray, mp: dict, mask_labeled: np.ndarray):
    idx = np.where(mask_labeled)[0]
    yt = pd.Series(y_full.iloc[idx]).dropna()
    if yt.empty or yt.nunique() < 2:
        return np.nan, np.nan, "Not enough samples or unique labels for evaluation", []
    yp = pd.Series(cluster_labels[idx], index=yt.index).map(mp)
    valid = ~yp.isna()
    yt, yp = yt[valid].astype(int), yp[valid].astype(int)
    if len(yt) == 0 or yt.nunique() < 2:
        return np.nan, np.nan, "Not enough samples or unique labels for evaluation", []
    acc = accuracy_score(yt, yp)
    prec = precision_score(yt, yp, average='macro', zero_division=0)
    classes = np.unique(yt)
    cm = confusion_matrix(yt, yp, labels=classes)
    return acc, prec, cm, classes

# ============== Preparación de datasets (RAW y PCA15) ==============
DF_RAW   = x_total_filtrado
DF_PCA15 = combined_x_pca_df.iloc[:, :15]
# asegurar mismo índice (derivan del mismo filtrado)
DF_PCA15 = DF_PCA15.reindex(DF_RAW.index)

# Máscara y etiquetas alineadas a DF_RAW
mask_labeled, y_train_full = build_mask_and_labels(DF_RAW, y_entrena_df)

dataframes_for_kmeans = {
    'x_total_filtrado': DF_RAW,
    'combined_x_pca_df_15_cols': DF_PCA15
}

k_range_kmeans = range(2, 6)  # 2..5
kmeans_results = {}

# carpeta de figuras
if 'figs' not in globals():
    figs = "./figs"
os.makedirs(figs, exist_ok=True)

# ============== Loop por dataset ==============
for df_name, df in dataframes_for_kmeans.items():
    print(f"Processing dataframe: {df_name}")

    X = df.to_numpy(dtype=float)

    inertia_scores = []
    silhouette_scores = []
    kmeans_models = {}

    # 1) Barrido K
    for k in k_range_kmeans:
        print(f"  Training KMeans for k = {k}")
        km = KMeans(n_clusters=k, init='k-means++', n_init=10, max_iter=100, tol=1e-4, random_state=42)
        km.fit(X)

        inertia_scores.append(km.inertia_)
        kmeans_models[k] = km

        labs = km.labels_
        if len(np.unique(labs)) >= 2:
            s = silhouette_score(X, labs, metric='euclidean')
            silhouette_scores.append(s)
            print(f"    Silhouette Score: {s:.4f}")
        else:
            silhouette_scores.append(np.nan)
            print("    Silhouette Score: N/A (less than 2 clusters)")

    # 3) Gráficos
    # Codo
    plt.figure(figsize=(10, 6))
    plt.plot(list(k_range_kmeans), inertia_scores, marker='o', linestyle='--')
    plt.xlabel('Número de Clusters (k)')
    plt.ylabel('Inercia')
    plt.grid(True)
    elbow_plot_path = os.path.join(figs, f'kmeans_elbow_{df_name}.png')
    plt.savefig(elbow_plot_path, dpi=150)
    print(f"  Elbow method plot saved to: {elbow_plot_path}")
    plt.show()

    # Silhouette
    if any(~np.isnan(silhouette_scores)):
        plt.figure(figsize=(10, 6))
        plt.plot(list(k_range_kmeans), silhouette_scores, marker='o', linestyle='--')
        plt.xlabel('Número de Clusters (k)')
        plt.ylabel('Coeficiente de Silhouette Promedio')
        plt.grid(True)
        silhouette_plot_path = os.path.join(figs, f'kmeans_silhouette_{df_name}.png')
        plt.savefig(silhouette_plot_path, dpi=150)
        print(f"  Silhouette score plot saved to: {silhouette_plot_path}")
        plt.show()
    else:
        print(f"  Skipping Silhouette plot for {df_name}: No valid silhouette scores.")

    # 4) Mejor k por silhouette
    valid_sil = [(kk, s) for kk, s in zip(list(k_range_kmeans), silhouette_scores) if not np.isnan(s)]
    if valid_sil:
        best_k, best_s = max(valid_sil, key=lambda kv: kv[1])
        best_inertia = inertia_scores[list(k_range_kmeans).index(best_k)]
        best_kmeans_model = kmeans_models[best_k]
        print(f"\n  Best k for {df_name} based on Silhouette: {best_k} (Silhouette: {best_s:.4f})")

        cluster_labels = best_kmeans_model.labels_

        # Voto mayoritario SOLO en filas etiquetadas
        cluster_to_label_mapping = majority_vote_mapping(y_train_full, cluster_labels, mask_labeled)
        print(f"  Cluster to Majority Label Mapping for {df_name}: {cluster_to_label_mapping}")

        # 5) Métricas sobre filas etiquetadas
        accuracy, precision_macro, conf_matrix, classes = eval_on_mask(y_train_full, cluster_labels, cluster_to_label_mapping, mask_labeled)

        if isinstance(conf_matrix, np.ndarray):
            print(f"  Evaluation for {df_name} (Best k = {best_k}):")
            print(f"    Accuracy: {accuracy:.4f}")
            print(f"    Precision (Macro): {precision_macro:.4f}")
            print(f"    Confusion Matrix:\n{conf_matrix}")

            # accuracy por clase
            print("    Accuracy per Class:")
            for i, class_label in enumerate(classes):
                row_sum = conf_matrix[i, :].sum()
                class_acc = (conf_matrix[i, i] / row_sum) if row_sum > 0 else np.nan
                print(f"      Class {class_label}: {class_acc:.4f}" if not np.isnan(class_acc) else f"      Class {class_label}: N/A")

            # Heatmap CM
            plt.figure(figsize=(8, 6))
            sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                        xticklabels=classes, yticklabels=classes)
            plt.xlabel('Predicted Label')
            plt.ylabel('True Label')
            conf_matrix_path = os.path.join(figs, f'kmeans_confusion_matrix_{df_name}.png')
            plt.savefig(conf_matrix_path, dpi=150)
            print(f"  Confusion matrix plot saved to: {conf_matrix_path}")
            plt.show()
        else:
            print(f"  Skipping evaluation for {df_name} (Best k = {best_k}): {conf_matrix}")

        # 6) Resumen por dataset
        summary = {
            'best_k': best_k,
            'silhouette': best_s,
            'inertia': best_inertia,
            'accuracy': accuracy if not np.isnan(accuracy) else np.nan,
            'precision_macro': precision_macro if not np.isnan(precision_macro) else np.nan,
            'cluster_to_label_mapping': cluster_to_label_mapping,
            'confusion_matrix': conf_matrix
        }
        kmeans_results[df_name] = summary
        print(f"\nSummary for {df_name}:")
        display(summary)
    else:
        print(f"\n  Could not determine best k for {df_name} based on Silhouette (no valid scores).")
        kmeans_results[df_name] = "No valid silhouette scores to determine best k"

print("\nK-Means evaluation complete for all dataframes.")
print("\nFinal K-Means Results Dictionary:")
display(kmeans_results)


In [ ]:
# CELDA 020
'''
Generar agrupamiento jerárquico con las siguientes condiciones:

1. Trabajar con:
    - x_total_filtrado
    - combined_x_pca_df (usar solo las primeras 15 columnas).
    - Serie/columna y_entrena, con índice compatible para cruce por índice (left join).

2. protoclusters (m) = 400. Se construyen con:
    - mediante MiniBatchKMeans
    - n_clusters=400
    - init='k-means++'
    - n_init=10
    - max_iter=100
    - tol=1e-4
    - random_state=42.

3. Guardar:
    - protoclusters = matriz (400 × d) de centroides.
    - proto_labels_full = índice de prototipo asignado a cada fila
      (BMU por distancia euclidiana).

4. Realizar cluster jerarquico sobre prototipos:
    - Probar enlaces ward (euclidiana) y average (UPGMA)
    - Cortar el árbol en k 2-5
    - Asignar etiqueta de k a los 9782 registros
    - Calcular silhouette promedio
    - En las filas labeled (mask_labeled), asigna a cada clúster la etiqueta
      mayoritaria (voto mayoritario) de acuerdo al cruce de los dataframes
      implementados con y_entrena

5. Evalúa sobre las filas etiquetadas:
    - Exactitud (accuracy) = proporción de aciertos
    - Precisión (precision macro) = promedio no ponderado de precisión por clase
      (usa average='macro').
    - Reporta también la matriz de confusión (también gráfico)

6. Entrega un resumen (imprime pero guarda en una variable) final por dataset:
   mejor k, silhouette, inercia, accuracy, precision macro y el diccionario
   cluster -> etiqueta.
'''

from sklearn.cluster import MiniBatchKMeans
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import silhouette_score, accuracy_score, precision_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import os

# ========= helpers mínimos =========
def build_mask_and_labels(x_filtrado: pd.DataFrame, y_entrena_df: pd.DataFrame | pd.Series):
    """
    Crea:
      - mask_labeled (n,): True si la fila proviene de x_entrena_df y sobrevivió al filtrado IQR.
      - y_train_full: Series (n,) con etiquetas en filas 'train' y NaN en 'test'.
    Supone: x_total = concat([x_entrena_df, x_prueba_df]).reset_index(drop=True)
            y luego x_total_filtrado = x_total[mask_outliers]  (SIN reset_index).
    """
    n_train = len(y_entrena_df)
    mask_labeled = (np.asarray(x_filtrado.index) < n_train)  # ndarray bool

    y_vals = pd.Series(y_entrena_df.squeeze()).to_numpy()
    y_train_full = pd.Series(np.nan, index=x_filtrado.index, name="clase")
    survived_train_idx = x_filtrado.index[mask_labeled]      # índices originales (< n_train) que sobrevivieron
    y_train_full.loc[survived_train_idx] = y_vals[survived_train_idx]
    return mask_labeled, y_train_full

def majority_vote_mapping(y_full: pd.Series, cluster_labels: np.ndarray, mask_labeled: np.ndarray) -> dict:
    idx = np.where(mask_labeled)[0]
    dfm = pd.DataFrame({"c": cluster_labels[idx], "y": y_full.iloc[idx].values}).dropna(subset=["y"])
    if dfm.empty:
        return {}
    return dfm.groupby("c")["y"].apply(lambda s: s.value_counts().idxmax()).to_dict()

def eval_on_mask(y_full: pd.Series, cluster_labels: np.ndarray, mp: dict, mask_labeled: np.ndarray):
    idx = np.where(mask_labeled)[0]
    yt = pd.Series(y_full.iloc[idx]).dropna()
    if yt.empty or yt.nunique() < 2:
        return np.nan, np.nan, "Not enough samples or unique labels for evaluation", []
    yp = pd.Series(cluster_labels[idx], index=yt.index).map(mp)
    valid = ~yp.isna()
    yt, yp = yt[valid].astype(int), yp[valid].astype(int)
    if len(yt) == 0 or yt.nunique() < 2:
        return np.nan, np.nan, "Not enough samples or unique labels for evaluation", []
    acc = accuracy_score(yt, yp)
    prec = precision_score(yt, yp, average='macro', zero_division=0)
    classes = np.unique(yt)
    cm = confusion_matrix(yt, yp, labels=classes)
    return acc, prec, cm, classes

def plot_cm(cm, classes, path_png):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes)
    plt.xlabel('Etiqueta predicha')
    plt.ylabel('Etiqueta real')
    plt.tight_layout()
    plt.savefig(path_png, dpi=150)
    print(f"        Confusion matrix plot saved to: {path_png}")
    plt.show()
    plt.close()

# ========= datasets de trabajo =========
DF_RAW   = x_total_filtrado
DF_PCA15 = combined_x_pca_df.iloc[:, :15].reindex(DF_RAW.index)
mask_labeled, y_train_full = build_mask_and_labels(DF_RAW, y_entrena_df)

dataframes_for_hierarchical = {
    'x_total_filtrado': DF_RAW,
    'combined_x_pca_df_15_cols': DF_PCA15
}

# ========= protoclústeres MBKMeans =========
m_protoclusters = 400
mbkmeans_params = {
    'n_clusters': m_protoclusters,
    'init': 'k-means++',
    'n_init': 10,
    'max_iter': 100,
    'tol': 1e-4,
    'random_state': 42,
    'batch_size': 2048
}

# ========= jerárquico =========
linkage_methods = ['ward', 'average']
k_range_hierarchical = range(2, 6)  # k = 2..5

hierarchical_results = {}
if 'figs' not in globals():
    figs = "./figs"
os.makedirs(figs, exist_ok=True)

# ========= loop por dataset =========
for df_name, df in dataframes_for_hierarchical.items():
    print(f"Processing dataframe for Hierarchical Clustering: {df_name}")
    X = df.to_numpy(dtype=float)

    # MiniBatchKMeans → protoclústeres
    print(f"  Creating {m_protoclusters} protoclusters using MiniBatchKMeans...")
    mbkmeans = MiniBatchKMeans(**mbkmeans_params)
    mbkmeans.fit(X)

    protoclusters    = mbkmeans.cluster_centers_            # (400 x d)
    proto_labels_full = mbkmeans.labels_.astype(int)        # (n,)
    inercia = float(mbkmeans.inertia_)                      # <- para el resumen

    print(f"  Protoclusters shape: {protoclusters.shape}")
    print(f"  Proto_labels_full shape: {proto_labels_full.shape}")

    method_results = []

    for method in linkage_methods:
        print(f"\n  Processing linkage method: {method}")
        Z = linkage(protoclusters, method=method, metric='euclidean')

        for k in k_range_hierarchical:
            print(f"    Evaluating k = {k}")
            protocluster_labels = fcluster(Z, k, criterion='maxclust')   # (m,), valores 1..k

            # Propagación a TODAS las filas
            propagated_labels = protocluster_labels[proto_labels_full]    # (n,)

            # Silhouette global (si hay ≥ 2 clústeres)
            if len(np.unique(propagated_labels)) >= 2:
                try:
                    silhouette_avg = float(silhouette_score(X, propagated_labels, metric='euclidean'))
                    print(f"      Silhouette Score: {silhouette_avg:.4f}")
                except Exception as e:
                    print(f"      Error calculating Silhouette for k={k}, method={method}: {e}")
                    silhouette_avg = np.nan
            else:
                print("      Silhouette Score: N/A (less than 2 clusters)")
                silhouette_avg = np.nan

            # Voto mayoritario y métricas SOLO en filas etiquetadas (mask_labeled)
            cluster_to_label_mapping = majority_vote_mapping(y_train_full, propagated_labels, mask_labeled)
            accuracy, precision_macro, conf_matrix, classes = eval_on_mask(
                y_train_full, propagated_labels, cluster_to_label_mapping, mask_labeled
            )

            if isinstance(conf_matrix, np.ndarray):
                print(f"      Evaluation for k = {k}, method = {method}:")
                print(f"        Accuracy: {accuracy:.4f}")
                print(f"        Precision (Macro): {precision_macro:.4f}")
                print(f"        Confusion Matrix:\n{conf_matrix}")

                # accuracy por clase
                print("        Accuracy per Class:")
                for i, class_label in enumerate(classes):
                    row_sum = conf_matrix[i, :].sum()
                    class_acc = (conf_matrix[i, i] / row_sum) if row_sum > 0 else np.nan
                    print(f"          Class {class_label}: {class_acc:.4f}" if not np.isnan(class_acc) else f"          Class {class_label}: N/A")

                # plot CM
                cm_path = os.path.join(figs, f'hierarchical_confusion_matrix_{df_name}_k{k}_{method}.png')
                plot_cm(conf_matrix, classes, cm_path)
            else:
                print(f"      Skipping evaluation for k={k}, method={method}: {conf_matrix}")

            method_results.append({
                'k': k,
                'method': method,
                'silhouette': silhouette_avg,
                'accuracy': accuracy,
                'precision_macro': precision_macro,
                'cluster_to_label_mapping': cluster_to_label_mapping,
                'confusion_matrix': conf_matrix,
                'inercia': inercia  # guardamos también para el resumen
            })

    # Resumen por dataset
    print(f"\nSummary for Hierarchical Clustering on {df_name}:")
    results_df = pd.DataFrame(method_results)

    if results_df['silhouette'].notna().any():
        best_idx = results_df['silhouette'].idxmax()
    else:
        best_idx = 0
    best = results_df.loc[best_idx]

    summary = {
        'best_k': int(best['k']),
        'best_method': str(best['method']),
        'silhouette': float(best['silhouette']) if pd.notna(best['silhouette']) else np.nan,
        'inercia': float(best['inercia']),
        'accuracy': float(best['accuracy']) if pd.notna(best['accuracy']) else np.nan,
        'precision_macro': float(best['precision_macro']) if pd.notna(best['precision_macro']) else np.nan,
        'cluster_to_label_mapping': dict(best['cluster_to_label_mapping']),
        'confusion_matrix': best['confusion_matrix']
    }
    hierarchical_results[df_name] = summary

    print("  Best Configuration:")
    display(summary)

print("\nHierarchical Clustering evaluation complete for all dataframes.")
print("\nFinal Hierarchical Clustering Results Dictionary:")
display(hierarchical_results)


In [ ]:
# CELDA 021
'''
Generar DBSCAN con las siguientes condiciones:

1. Trabajar con:
	- x_total_filtrado
	- combined_x_pca_df (usar solo las primeras 15 columnas).
	- Serie/columna y_entrena, con índice compatible para cruce por índice (left join).

2. Búsqueda DBSCAN (métrica euclidiana)
	- min_samples ∈ {5, 8, 10, 12, 16}.
	- Para cada min_samples,  barrer eps ∈ [4.5, 5.0] con paso 0.1.
	- Fijar semillas globales (cuando aplique) para reproducibilidad.

3. Para cada configuración DBSCAN probada guardar:
	- eps
	- min_samples
	- n_clusters excluyendo la etiqueta -1
	- noise_ratio = (# etiquetas -1) / N
	- silhouette (solo con puntos ≠ -1; si hay <2 clústeres válidos, guardar silhouette = NaN)
	- En las filas labeled (mask_labeled), asigna a cada clúster la etiqueta mayoritaria
    (voto mayoritario) de acuerdo al cruce de los dataframes implementados con y_entrena

5. Evalúar sobre las filas etiquetadas:

	- Exactitud (accuracy) = proporción de aciertos
	- Precisión (precision macro) = promedio no ponderado de precisión por clase (usa average='macro').
	- Reportar también la matriz de confusión

4. Construir una tabla con todas las configuraciones y ordenar por:

	- silhouette desc
	- noise_ratio asc
	- n_clusters desc
	- desempates por silhouette, luego eps, luego mayor min_samples (en ese orden).
'''

# PROTO(KMeans++) → DBSCAN(protos) → Propagar → Mayoría(mask) → Métricas (RAW y PCA15)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, accuracy_score, precision_score, confusion_matrix

# ================= Config =================
eps_grid = [round(e, 1) for e in np.arange(4.5, 5.0 + 1e-9, 0.1)]
ms_grid  = [5, 8, 10, 12, 16]
if 'figs' not in globals():
    figs = "./figs"
os.makedirs(figs, exist_ok=True)

# ================= Helpers =================
def sil_ignora_ruido(X: np.ndarray, lab_full: np.ndarray) -> float:
    m = lab_full != -1
    if m.sum() < 2: return np.nan
    u = np.unique(lab_full[m])
    return np.nan if len(u) < 2 else silhouette_score(X[m], lab_full[m], metric='euclidean')

def build_mask_and_labels(x_filtrado: pd.DataFrame, y_entrena_df: pd.DataFrame | pd.Series):
    """
    Crea:
      - mask_labeled (n,): True si la fila proviene de x_entrena_df y sobrevivió al filtrado IQR.
      - y_train_full: Series (n,) con etiquetas en filas 'train' y NaN en 'test'.
    Supone: x_total = concat([x_entrena_df, x_prueba_df]).reset_index(drop=True)
            y luego x_total_filtrado = x_total[mask_outliers]  (SIN reset_index).
    """
    n_train = len(y_entrena_df)
    mask_labeled = (np.asarray(x_filtrado.index) < n_train)  # ndarray bool

    y_vals = pd.Series(y_entrena_df.squeeze()).to_numpy()
    y_train_full = pd.Series(np.nan, index=x_filtrado.index, name="clase")
    survived_train_idx = x_filtrado.index[mask_labeled]
    y_train_full.loc[survived_train_idx] = y_vals[survived_train_idx]
    return mask_labeled, y_train_full

def majority_vote_mapping_mask(y_full: pd.Series, cluster_labels: np.ndarray, mask_labeled: np.ndarray) -> dict:
    idx = np.where(mask_labeled)[0]
    dfm = pd.DataFrame({'c': cluster_labels[idx], 'y': y_full.iloc[idx].values}).dropna(subset=['y'])
    if dfm.empty: return {}
    return dfm.groupby('c')['y'].apply(lambda s: s.value_counts().idxmax()).to_dict()

def eval_on_mask(y_full: pd.Series, cluster_labels: np.ndarray, mp: dict, mask_labeled: np.ndarray):
    idx = np.where(mask_labeled)[0]
    yt = pd.Series(y_full.iloc[idx]).dropna()
    if yt.empty or yt.nunique() < 2:
        return np.nan, np.nan, None
    yp = pd.Series(cluster_labels[idx], index=yt.index).map(mp).astype('object')
    v = ~yp.isna()
    if v.sum() == 0:
        return np.nan, np.nan, None
    yt, yp = yt[v].astype(str), yp[v].astype(str)
    acc = accuracy_score(yt, yp)
    prec = precision_score(yt, yp, average='macro', zero_division=0)
    labs = sorted(yt.unique())
    cm = pd.DataFrame(confusion_matrix(yt, yp, labels=labs),
                      index=[f"true_{c}" for c in labs],
                      columns=[f"pred_{c}" for c in labs])
    return acc, prec, cm

def sort_config_table(df):
    # Orden requerido: silhouette DESC, noise_ratio ASC, n_clusters DESC, luego eps, luego min_samples DESC
    return df.sort_values(
        by=['silhouette', 'noise_ratio', 'n_clusters', 'eps', 'min_samples'],
        ascending=[False, True, False, True, False]
    )

def plot_top4(df_sorted, title, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    top4 = df_sorted.head(4)
    if top4.empty:
        print(f"[AVISO] Sin configs para {title}"); return
    xs = [f"eps={r.eps:.1f}\nms={int(r.min_samples)}" for _, r in top4.iterrows()]
    ys = top4['silhouette'].fillna(0).values
    plt.figure(figsize=(6,4))
    plt.bar(range(len(ys)), ys)
    plt.xticks(range(len(ys)), xs); plt.ylabel("Silhouette (sin ruido)")
    plt.title(title); plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()

# ================= Entradas y máscara =================
# RAW y PCA15 con MISMA cantidad de filas (9782) y mismo índice (derivan de x_total_filtrado)
DF_RAW   = x_total_filtrado
DF_PCA15 = combined_x_pca_df.iloc[:, :15].reindex(DF_RAW.index)

mask_labeled, y_full = build_mask_and_labels(DF_RAW, y_entrena_df)

datasets = [('RAW', DF_RAW), ('PCA15', DF_PCA15)]
results, summaries = {}, {}

for name, Xdf in datasets:
    print(f"\n[{name}] N={len(Xdf)} | labeled={mask_labeled.sum()} | unlabeled={len(Xdf)-mask_labeled.sum()}")
    X = Xdf.to_numpy(dtype=float)

    # 1) Protoclústeres K-Means++ (m=400)
    km = KMeans(n_clusters=400, init='k-means++', n_init=10, max_iter=100, tol=1e-4, random_state=42)
    km.fit(X)
    protos, proto_lab_full = km.cluster_centers_, km.labels_  # (m,), para TODAS las filas

    # 2) Barrido DBSCAN sobre protoclústeres
    rows = []
    for ms in ms_grid:
        for eps in eps_grid:
            lab_protos = DBSCAN(eps=eps, min_samples=ms, metric='euclidean').fit_predict(protos)
            lab_full   = lab_protos[proto_lab_full]  # propagar a las N filas

            ncl   = int(len(np.unique(lab_protos[lab_protos != -1])))
            noise = float(np.mean(lab_full == -1))
            sil   = sil_ignora_ruido(X, lab_full)

            mp  = majority_vote_mapping_mask(y_full, lab_full, mask_labeled)
            acc, prec, _ = eval_on_mask(y_full, lab_full, mp, mask_labeled)

            rows.append({
                'eps': eps,
                'min_samples': ms,
                'n_clusters': ncl,
                'noise_ratio': noise,
                'silhouette': sil,
                'accuracy_labeled': acc,
                'precision_macro_labeled': prec
            })

    df = pd.DataFrame(rows)
    df_sorted = sort_config_table(df.copy())
    results[name] = df_sorted

    # === Tabla completa ordenada (impresión) ===
    print(f"\n[{name}] TODAS LAS CONFIGURACIONES (ordenadas):")
    print(df_sorted.to_string(index=False))

    # === Resumen por número de clústeres (promedios) ===
    print(f"\n[{name}] RESUMEN por n_clusters (promedios):")
    summary_by_k = df.groupby('n_clusters', dropna=False)[
        ['silhouette','noise_ratio','accuracy_labeled','precision_macro_labeled']
    ].mean().sort_index()
    print(summary_by_k)

    # 3) Mejor configuración por la tabla ordenada
    if df_sorted.empty:
        summaries[name] = {'msg':'sin configuraciones'}; continue

    best = df_sorted.iloc[0]
    lab_full_best = DBSCAN(eps=float(best.eps), min_samples=int(best.min_samples), metric='euclidean') \
                        .fit_predict(protos)[proto_lab_full]
    mp_best = majority_vote_mapping_mask(y_full, lab_full_best, mask_labeled)
    acc_b, prec_b, cm_b = eval_on_mask(y_full, lab_full_best, mp_best, mask_labeled)
    sil_b = sil_ignora_ruido(X, lab_full_best)
    noise_b = float(np.mean(lab_full_best == -1))

    summaries[name] = dict(
        best_eps=float(best.eps),
        best_min_samples=int(best.min_samples),
        n_clusters_protos=int(best.n_clusters),
        noise_ratio_filas=noise_b,
        silhouette_filas=sil_b,
        accuracy_labeled=acc_b,
        precision_macro_labeled=prec_b,
        cluster_to_label=mp_best,
        confusion_matrix_df=cm_b
    )

    # 4) Plot Top-4 por dataset
    plot_top4(df_sorted, f"DBSCAN-Protos Top-4 — {name}", os.path.join(figs, f"dbscan_protos_{name}_top4.png"))

# ================= Salidas finales =================
for k in ['RAW','PCA15']:
    print(f"\n== {k} | MEJOR CONFIG ==")
    for kk, vv in summaries.get(k, {}).items():
        if kk not in ('confusion_matrix_df','cluster_to_label'):
            print(f"{kk}: {vv}")
    print("cluster_to_label:", summaries.get(k, {}).get('cluster_to_label', {}))
    if summaries.get(k, {}).get('confusion_matrix_df') is not None:
        print("\nMatriz de confusión:\n", summaries[k]['confusion_matrix_df'])


In [ ]:
# CELDA 022
'''
Ejecutar y evalúar un SOM 30×30 (Self-Organizing Map) sobre dos datasets:

- X1: x_total_filtrado (todas las variables).
- X2: combined_x_pca_df[:, :15] (primeros 15 componentes principales).
- y_entrena_df: serie de etiquetas alineada por índice con valores {-1, 1}; -1 = no etiquetado.

Flujo:
1) Entrena SOM 30×30 por dataset con vecindad gaussiana en dos fases:
   - Rough (20 épocas): tasa de aprendizaje 0.4→0.3, radio de vecindad 15→8
   - Fine (80 épocas): tasa de aprendizaje 0.3→0.2, redio de vecindad 8→2
2) Asigna BMU por muestra y calcula el Quantization Error (QE).
3) Etiqueta neuronas por voto mayoritario usando solo registros con y≠-1.
4) Evalúa en el subconjunto etiquetado: Accuracy, Precision/Recall/F1 macro y Balanced Accuracy.
5) Genera y guarda la matriz de confusión (heatmap) con etiquetas {-1, 1} y muestra la ruta guardada.
6) Imprime resumen de métricas y listado de archivos en la carpeta de figuras.

'''

import numpy as np, pandas as pd, os, matplotlib.pyplot as plt
from minisom import MiniSom
from collections import Counter
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, balanced_accuracy_score

# ================== Parámetros y rutas ==================
if 'figs' not in globals():
    figs = "./figs"
os.makedirs(figs, exist_ok=True)

# ================== Máscara y etiquetas alineadas al filtrado ==================
def build_mask_and_labels(x_filtrado: pd.DataFrame, y_entrena_df: pd.DataFrame | pd.Series):
    """
    Crea:
      - mask_labeled (n,): True si la fila proviene de x_entrena_df y sobrevivió al filtrado IQR.
      - y_full: Series (n,) con etiqueta en filas 'train' y NaN en 'test' (no etiquetadas).
    Supone: x_total = concat([x_entrena_df, x_prueba_df]).reset_index(drop=True)
            y luego x_total_filtrado = x_total[mask_outliers]  (SIN reset_index).
    """
    n_train = len(y_entrena_df)
    mask_labeled = (np.asarray(x_filtrado.index) < n_train)  # ndarray bool

    y_vals = pd.Series(y_entrena_df.squeeze()).to_numpy()
    y_full = pd.Series(np.nan, index=x_filtrado.index, name="clase")
    survived_train_idx = x_filtrado.index[mask_labeled]
    y_full.loc[survived_train_idx] = y_vals[survived_train_idx]
    return mask_labeled, y_full

mask_labeled, y_full = build_mask_and_labels(x_total_filtrado, y_entrena_df)

# ================== Datos (RAW y PCA15) ==================
X1 = x_total_filtrado.to_numpy(dtype=float)
X2 = combined_x_pca_df.iloc[:, :15].reindex(x_total_filtrado.index).to_numpy(dtype=float)

n_total_1, d1 = X1.shape
n_total_2, d2 = X2.shape
assert n_total_1 == n_total_2 == len(mask_labeled), "Tamaños inconsistentes en X1/X2/mask"

# ================== Entrenamiento SOM (dos fases) ==================
def entrenar_som_30x30(X, seed=42):
    som = MiniSom(30, 30, X.shape[1], sigma=15.0, learning_rate=0.4,
                  neighborhood_function='gaussian', random_seed=seed)
    som.random_weights_init(X)
    # Rough (20): lr 0.4→0.3, sigma 15→8
    for lr, sg in zip(np.linspace(0.4, 0.3, 20), np.linspace(15.0, 8.0, 20)):
        som._learning_rate, som._sigma = float(lr), float(sg)
        som.train_batch(X, num_iteration=1)
    # Fine (80): lr 0.3→0.2, sigma 8→2
    for lr, sg in zip(np.linspace(0.3, 0.2, 80), np.linspace(8.0, 2.0, 80)):
        som._learning_rate, som._sigma = float(lr), float(sg)
        som.train_batch(X, num_iteration=1)
    return som

def bmu_idx(som, X):
    r, c = som.get_weights().shape[:2]
    return np.array([som.winner(x)[0]*c + som.winner(x)[1] for x in X], dtype=int)

def qe_score(som, X, bmu):
    W = som.get_weights().reshape(-1, X.shape[1])
    return float(np.mean(np.linalg.norm(X - W[bmu], axis=1)))

# ================== Etiquetado (voto mayoritario) y evaluación (con máscara) ==================
def etiquetar_neuronas_por_mayoria_mask(bmu, y_full, mask_labeled):
    """
    Mapa neuron_id -> etiqueta mayoritaria usando SOLO filas etiquetadas (mask_labeled==True).
    """
    m = int(bmu.max()) + 1
    grupos = [[] for _ in range(m)]
    idx_lab = np.where(mask_labeled)[0]
    for bi, yi in zip(bmu[idx_lab], y_full.iloc[idx_lab]):
        if pd.notna(yi):
            grupos[int(bi)].append(int(yi))
    return {i: (Counter(g).most_common(1)[0][0] if g else None) for i, g in enumerate(grupos)}

def evaluar_mask(bmu, y_full, lab_map, mask_labeled):
    """
    Métricas SOLO sobre filas etiquetadas (mask_labeled==True).
    """
    idx_lab = np.where(mask_labeled)[0]
    y_true = y_full.iloc[idx_lab].dropna()
    if y_true.empty:
        return None, np.nan, np.nan, np.nan, np.nan, np.nan, {}, []

    # predecir solo en las mismas posiciones de y_true
    # (alineamos por índice de y_true)
    bmu_lab = pd.Series(bmu[idx_lab], index=y_full.index[idx_lab]).loc[y_true.index].to_numpy()
    y_pred = np.array([lab_map.get(int(bi), None) for bi in bmu_lab], dtype=object)

    valid = (y_pred != None)
    y_true = y_true.iloc[valid].astype(int).to_numpy()
    y_pred = y_pred[valid].astype(int)
    if y_true.size == 0 or len(np.unique(y_true)) < 2:
        return None, np.nan, np.nan, np.nan, np.nan, np.nan, {}, []

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    bal  = balanced_accuracy_score(y_true, y_pred)
    classes_sorted = sorted(np.unique(y_true))
    cm = confusion_matrix(y_true, y_pred, labels=classes_sorted)

    # accuracy por clase
    acc_cls = {}
    for i, lab in enumerate(classes_sorted):
        fila = cm[i, :].sum()
        acc_cls[int(lab)] = (cm[i, i] / fila) if fila > 0 else np.nan

    return cm, acc, prec, rec, f1, bal, acc_cls, classes_sorted

# ================== Plot Matriz de Confusión ==================
def plot_confusion_matrix(cm, labels_sorted, title, out_name):
    plt.figure(figsize=(6.5, 5.5))
    plt.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.title(title)
    plt.colorbar()
    ticks = np.arange(len(labels_sorted))
    plt.xticks(ticks, labels_sorted); plt.yticks(ticks, labels_sorted)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha='center', va='center')
    plt.xlabel('Predicted Label'); plt.ylabel('True Label')
    out_path = os.path.join(figs, out_name)
    plt.tight_layout()
    plt.savefig(out_path, dpi=140)
    plt.show()
    plt.close()
    print(f"[Guardado] {out_path}")
    return out_path

# ================== X1 (RAW) ==================
som1 = entrenar_som_30x30(X1, seed=42)
b1   = bmu_idx(som1, X1)
qe1  = qe_score(som1, X1, b1)
lab1 = etiquetar_neuronas_por_mayoria_mask(b1, y_full, mask_labeled)
cm1, acc1, prec1, rec1, f11, bal1, acc_cls1, labs1 = evaluar_mask(b1, y_full, lab1, mask_labeled)
if cm1 is not None:
    path1 = plot_confusion_matrix(cm1, labs1, "", "confusion_matrix_X1.png")
else:
    path1 = "(no disponible)"

print(f"""
        Accuracy: {acc1 if not np.isnan(acc1) else float('nan'):.4f}
        Precision (Macro): {prec1 if not np.isnan(prec1) else float('nan'):.4f}
        Recall (Macro): {rec1 if not np.isnan(rec1) else float('nan'):.4f}
        F1 (Macro): {f11 if not np.isnan(f11) else float('nan'):.4f}
        Balanced Accuracy: {bal1 if not np.isnan(bal1) else float('nan'):.4f}
        Confusion Matrix:
{cm1 if cm1 is not None else 'N/A'}
        Accuracy per Class:
{acc_cls1}
        Confusion matrix plot saved to: {path1}
""".rstrip())

# ================== X2 (PCA15) ==================
som2 = entrenar_som_30x30(X2, seed=42)
b2   = bmu_idx(som2, X2)
qe2  = qe_score(som2, X2, b2)
lab2 = etiquetar_neuronas_por_mayoria_mask(b2, y_full, mask_labeled)
cm2, acc2, prec2, rec2, f12, bal2, acc_cls2, labs2 = evaluar_mask(b2, y_full, lab2, mask_labeled)
if cm2 is not None:
    path2 = plot_confusion_matrix(cm2, labs2, "", "confusion_matrix_X2.png")
else:
    path2 = "(no disponible)"

print(f"""
        Accuracy: {acc2 if not np.isnan(acc2) else float('nan'):.4f}
        Precision (Macro): {prec2 if not np.isnan(prec2) else float('nan'):.4f}
        Recall (Macro): {rec2 if not np.isnan(rec2) else float('nan'):.4f}
        F1 (Macro): {f12 if not np.isnan(f12) else float('nan'):.4f}
        Balanced Accuracy: {bal2 if not np.isnan(bal2) else float('nan'):.4f}
        Confusion Matrix:
{cm2 if cm2 is not None else 'N/A'}
        Accuracy per Class:
{acc_cls2}
        Confusion matrix plot saved to: {path2}
""".rstrip())

# ================== Resumen QE y listado de archivos ==================
res = pd.DataFrame([
    dict(dataset="X1_RAW",  grid="30x30", neurons=900, QE=qe1, acc=acc1, prec=prec1, rec=rec1, f1=f11),
    dict(dataset="X2_PCA15",grid="30x30", neurons=900, QE=qe2, acc=acc2, prec=prec2, rec=rec2, f1=f12),
]).sort_values("QE", ascending=True)

print("\n=== Resultados (ordenado por QE ascendente) ===")
print(res.to_string(index=False))

print("\nContenido de la carpeta de figuras:")
for f in sorted(os.listdir(figs)):
    print(" -", os.path.join(figs, f))


In [54]:
# === SOM sobre PCA15 para etiquetar x_prueba_df y exportar CSV (20 features + label_final) ===
import numpy as np, pandas as pd, os
from minisom import MiniSom
from collections import Counter
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score, confusion_matrix

# ------------------ 1) Preparación (sin filtrado) ------------------
# Se asume que ya existen: x_entrena_df, x_prueba_df, y_entrena_df, x_total (concat de ambos)
n_train = len(y_entrena_df)
n_total = len(x_total)
assert n_total == len(x_entrena_df) + len(x_prueba_df), "x_total debe ser concat([x_entrena_df, x_prueba_df])."
mask_labeled = (np.arange(n_total) < n_train)  # primeras n_train filas = entrenamiento (no hay filtrado)

# Serie de etiquetas alineada a x_total: y en train, NaN en test
y_full = pd.Series(np.nan, index=np.arange(n_total), name="clase")
y_full.iloc[:n_train] = pd.Series(y_entrena_df.squeeze()).values

# ------------------ 2) PCA en x_total y toma de 15 PCs ------------------
pca = PCA(n_components=None, random_state=42)
X_total = x_total.to_numpy(dtype=float)
combined_x_pca = pca.fit_transform(X_total)
combined_x_pca_df = pd.DataFrame(combined_x_pca, index=x_total.index)  # por si quieres reutilizarlo luego
X_pca15 = combined_x_pca_df.iloc[:, :15].to_numpy(dtype=float)         # solo 15 PCs

# ------------------ 3) Entrenamiento SOM 30×30 (rough+fine) ------------------
def entrenar_som_30x30(X, seed=42):
    som = MiniSom(30, 30, X.shape[1], sigma=15.0, learning_rate=0.4,
                  neighborhood_function='gaussian', random_seed=seed)
    som.random_weights_init(X)
    # Rough: 20 pasos (lr 0.4→0.3, sigma 15→8)
    for lr, sg in zip(np.linspace(0.4, 0.3, 20), np.linspace(15.0, 8.0, 20)):
        som._learning_rate, som._sigma = float(lr), float(sg)
        som.train_batch(X, num_iteration=1)
    # Fine: 80 pasos (lr 0.3→0.2, sigma 8→2)
    for lr, sg in zip(np.linspace(0.3, 0.2, 80), np.linspace(8.0, 2.0, 80)):
        som._learning_rate, som._sigma = float(lr), float(sg)
        som.train_batch(X, num_iteration=1)
    return som

def bmu_idx(som, X):
    r, c = som.get_weights().shape[:2]
    return np.array([som.winner(x)[0]*c + som.winner(x)[1] for x in X], dtype=int)

def qe_score(som, X, bmu):
    W = som.get_weights().reshape(-1, X.shape[1])
    return float(np.mean(np.linalg.norm(X - W[bmu], axis=1)))

som = entrenar_som_30x30(X_pca15, seed=42)
bmu = bmu_idx(som, X_pca15)
qe  = qe_score(som, X_pca15, bmu)

# ------------------ 4) Voto mayoritario por neurona (solo train) ------------------
def etiqueta_por_neurona(bmu, y_full, mask_labeled):
    m = int(bmu.max()) + 1
    grupos = [[] for _ in range(m)]
    idx_lab = np.where(mask_labeled)[0]
    for bi, yi in zip(bmu[idx_lab], y_full.iloc[idx_lab]):
        if pd.notna(yi):
            grupos[int(bi)].append(int(yi))
    mp = {i: (Counter(g).most_common(1)[0][0] if g else None) for i, g in enumerate(grupos)}
    # fallback: mayoría global si quedan neuronas sin votos
    if any(v is None for v in mp.values()):
        if y_full.notna().any():
            global_major = int(pd.Series(y_full.iloc[:n_train]).dropna().mode().iloc[0])
            for k in mp:
                if mp[k] is None:
                    mp[k] = global_major
    return mp

mp = etiqueta_por_neurona(bmu, y_full, mask_labeled)

# Predicciones SOM para todas las filas
som_pred = np.array([mp.get(int(bi)) for bi in bmu], dtype=int)

# ------------------ 5) Métricas en el bloque etiquetado (solo consola) ------------------
y_true_tr = y_full.iloc[:n_train].astype(int).to_numpy()
y_pred_tr = som_pred[:n_train]

acc  = accuracy_score(y_true_tr, y_pred_tr)
prec = precision_score(y_true_tr, y_pred_tr, average='macro', zero_division=0)
rec  = recall_score(y_true_tr, y_pred_tr, average='macro', zero_division=0)
f1   = f1_score(y_true_tr, y_pred_tr, average='macro', zero_division=0)
bal  = balanced_accuracy_score(y_true_tr, y_pred_tr)
classes_sorted = sorted(np.unique(y_true_tr))
cm   = confusion_matrix(y_true_tr, y_pred_tr, labels=classes_sorted)

print("\n=== Métricas SOM (PCA15) en entrenamiento ===")
print(f"QE (Quantization Error): {qe:.6f}")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f} (macro)")
print(f"Recall:    {rec:.4f} (macro)")
print(f"F1:        {f1:.4f} (macro)")
print(f"Balanced Accuracy: {bal:.4f}")
print("Confusion Matrix (orden de clases = {}):".format(classes_sorted))
print(cm)

# ------------------ 6) CSV final (solo x_prueba_df: 20 features + label_final) ------------------
# Etiqueta final: en train = etiqueta original; en test = pred SOM
label_final = np.empty(n_total, dtype=int)
label_final[:n_train] = y_true_tr
label_final[n_train:] = som_pred[n_train:]

# Armar salida SOLO para x_prueba_df (las últimas len(x_prueba_df) filas de x_total)
start_test = n_train
end_test   = n_total
out_df = x_prueba_df.copy()                # 20 features originales
out_df['label_final'] = label_final[start_test:end_test]

# Guardar CSV en la ruta `data`
os.makedirs(data, exist_ok=True)
filename = f"som_pca15_labels_x_prueba_{len(out_df)}x{out_df.shape[1]}.csv"  # 10000 x 21 típico
datos = os.path.join(data, filename)
out_df.to_csv(datos, index=False)

print(f"\n[OK] CSV de x_prueba_df con etiqueta SOM guardado en: {datos}")
print(f"Forma: {out_df.shape[0]} filas x {out_df.shape[1]} columnas (20 features + label_final)")



=== Métricas SOM (PCA15) en entrenamiento ===
QE (Quantization Error): 3.464609
Accuracy:  0.7920
Precision: 0.7926 (macro)
Recall:    0.7893 (macro)
F1:        0.7902 (macro)
Balanced Accuracy: 0.7893
Confusion Matrix (orden de clases = [np.int64(-1), np.int64(1)]):
[[350 119]
 [ 89 442]]

[OK] CSV de x_prueba_df con etiqueta SOM guardado en: /content/t7drive/MyDrive/taller7/datos/som_pca15_labels_x_prueba_10000x21.csv
Forma: 10000 filas x 21 columnas (20 features + label_final)
